# Taller 2 — DQN sobre LunarLander-v3

**Simulación y Aprendizaje por Refuerzo** — Maestría en IA, Universidad de La Sabana.  
Autor: Ivan Enrique Rangel Santos.

Notebook autónomo para entrenar un agente DQN sobre LunarLander-v3 en Google Colab.

Antes de ejecutar: **Runtime → Change runtime type → T4 GPU** (aunque LunarLander corre bien también en CPU, GPU acelera ~3×).

Duración: **~10–20 minutos** para 300 000 pasos.

## 1. Instalación

In [ ]:
!pip install -q swig "gymnasium[box2d]==1.3.0" "torch" "numpy<2" "matplotlib" "imageio[ffmpeg]"

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Montar Google Drive (para no perder los checkpoints si Colab se corta)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/lunarlander_dqn'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Los checkpoints se guardarán en:', DRIVE_DIR)

## 3. El ambiente LunarLander-v3

**Objetivo:** aterrizar suavemente un módulo lunar en el pad marcado entre dos banderas, sin estrellarse.

**Observación:** vector de 8 dimensiones continuas — mucho más rico que MountainCar (2) pero más pequeño que Atari (imagen).

| Índice | Componente | Rango típico |
|---|---|---|
| 0 | `x` — posición horizontal | [-1.5, 1.5] |
| 1 | `y` — posición vertical | [0, 1.5] |
| 2 | `vx` — velocidad horizontal | [-5, 5] |
| 3 | `vy` — velocidad vertical | [-5, 5] |
| 4 | `angle` — orientación | [-π, π] |
| 5 | `ω` — velocidad angular | [-5, 5] |
| 6 | `contacto_pata_izq` | {0, 1} |
| 7 | `contacto_pata_der` | {0, 1} |

**Acciones:** 4 discretas.

| Índice | Acción | Efecto |
|---|---|---|
| 0 | NADA | Deja actuar la gravedad |
| 1 | motor izquierdo | Rota a la derecha |
| 2 | motor principal | Empuja hacia arriba |
| 3 | motor derecho | Rota a la izquierda |

**Recompensa (densa, con múltiples componentes):**
- Acercarse al pad: gradiente positivo entre −100 y +100 según distancia y velocidad.
- Cada pata haciendo contacto: +10.
- Aterrizaje suave: +100 al final.
- Crash: −100.
- Costo por combustible: −0.3/frame usando motor principal, −0.03/frame usando laterales.

**Umbral "resuelto" oficial:** retorno promedio de **+200** sobre 100 episodios consecutivos.

**Terminación:** aterrizaje, crash, o corte a 1000 pasos (`truncated`).

In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make('LunarLander-v3')
obs, _ = env.reset(seed=0)
print('obs shape:', obs.shape, 'dtype:', obs.dtype)
print('obs sample:', obs)
print('action space:', env.action_space)
env.close()

## 4. Red neuronal — MLP

```
Input:  (batch, 8) float32          # las 8 componentes del estado
Linear(8   → 128) → ReLU
Linear(128 → 128) → ReLU
Linear(128 →   4)                    # 1 Q por acción
```

**Por qué MLP y no CNN:** el estado ya es un vector de features estructuradas (posición, velocidad, ángulo…). No hay imagen. Una CNN aquí no aportaría nada porque las 8 componentes no tienen relación espacial 2D. Un MLP es lo natural.

**Por qué 128 neuronas por capa:** compromiso estándar. Suficiente capacidad para modelar la función Q sobre 8 entradas continuas, no tan grande que sobreajuste transitorios del entrenamiento. Redes más pequeñas (64) también funcionan pero convergen más lento; más grandes (256) son innecesarias.

**Dos capas ocultas:** una sola no basta porque la función Q depende de interacciones no triviales entre componentes (por ejemplo, el efecto de encender el motor principal depende del ángulo actual). Tres capas o más no dan ventaja consistente en este problema.

In [ ]:
import torch
import torch.nn as nn

class QNetwork(nn.Module):
    def __init__(self, state_dim=8, n_actions=4, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),    nn.ReLU(inplace=True),
            nn.Linear(hidden, n_actions),
        )
    def forward(self, x):
        return self.net(x)

m = QNetwork()
print('parámetros:', sum(p.numel() for p in m.parameters()))
print('forward:', m(torch.zeros(2, 8)).shape)

## 5. Replay buffer

Almacenamiento en `float32` — como el estado es solo 8 números, no hay presión de memoria como con Atari. 100 000 × 8 × 4 bytes ≈ 3.2 MB para los estados. Trivial.

In [ ]:
from collections import namedtuple

Transition = namedtuple('Transition', 'obs actions rewards next_obs dones')

class ReplayBuffer:
    def __init__(self, capacity, obs_dim, device='cpu'):
        self.capacity = int(capacity); self.device = device
        self.obs      = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions  = np.zeros(capacity, dtype=np.int64)
        self.rewards  = np.zeros(capacity, dtype=np.float32)
        self.dones    = np.zeros(capacity, dtype=np.float32)
        self._idx = 0; self._size = 0

    def __len__(self): return self._size

    def push(self, obs, a, r, next_obs, done):
        i = self._idx
        self.obs[i] = obs; self.next_obs[i] = next_obs
        self.actions[i] = a; self.rewards[i] = r; self.dones[i] = float(done)
        self._idx = (i + 1) % self.capacity
        self._size = min(self._size + 1, self.capacity)

    def sample(self, batch_size):
        idx = np.random.randint(0, self._size, size=batch_size)
        d = self.device
        return Transition(
            torch.from_numpy(self.obs[idx]).to(d),
            torch.from_numpy(self.actions[idx]).to(d),
            torch.from_numpy(self.rewards[idx]).to(d),
            torch.from_numpy(self.next_obs[idx]).to(d),
            torch.from_numpy(self.dones[idx]).to(d),
        )

## 6. Agente DQN

In [ ]:
from copy import deepcopy
from dataclasses import dataclass
import torch.nn.functional as F

@dataclass
class DQNConfig:
    env_id: str = 'LunarLander-v3'
    state_dim: int = 8
    n_actions: int = 4
    hidden: int = 128
    lr: float = 5e-4
    gamma: float = 0.99
    batch_size: int = 64
    max_grad_norm: float = 10.0
    eps_start: float = 1.0
    eps_end: float = 0.02
    eps_decay_steps: int = 100_000
    buffer_capacity: int = 100_000
    replay_start_size: int = 1_000
    learn_every: int = 1
    target_update_every: int = 500
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    seed: int = 0

class DQNAgent:
    def __init__(self, cfg):
        self.cfg = cfg
        torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
        self.q_net      = QNetwork(cfg.state_dim, cfg.n_actions, cfg.hidden).to(cfg.device)
        self.target_net = deepcopy(self.q_net).to(cfg.device)
        for p in self.target_net.parameters(): p.requires_grad = False
        self.optimizer  = torch.optim.Adam(self.q_net.parameters(), lr=cfg.lr)
        self.buffer     = ReplayBuffer(cfg.buffer_capacity, cfg.state_dim, cfg.device)
        self.step_count = 0

    def epsilon(self):
        frac = min(1.0, self.step_count / self.cfg.eps_decay_steps)
        return self.cfg.eps_start + frac * (self.cfg.eps_end - self.cfg.eps_start)

    @torch.no_grad()
    def select_action(self, obs, deterministic=False):
        if not deterministic and np.random.random() < self.epsilon():
            return int(np.random.randint(self.cfg.n_actions))
        obs_t = torch.from_numpy(obs).unsqueeze(0).to(self.cfg.device)
        return int(self.q_net(obs_t).argmax(1).item())

    def learn(self):
        if len(self.buffer) < self.cfg.replay_start_size:
            return None
        batch = self.buffer.sample(self.cfg.batch_size)
        current_q = self.q_net(batch.obs).gather(1, batch.actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q = self.target_net(batch.next_obs).max(1).values
            target_q = batch.rewards + self.cfg.gamma * next_q * (1.0 - batch.dones)
        loss = F.smooth_l1_loss(current_q, target_q)
        self.optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), self.cfg.max_grad_norm)
        self.optimizer.step()
        return float(loss.item())

    def sync_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())

    def save(self, path):
        torch.save({'cfg': self.cfg.__dict__,
                    'q_net': self.q_net.state_dict(),
                    'target_net': self.target_net.state_dict(),
                    'optimizer': self.optimizer.state_dict(),
                    'step_count': self.step_count}, path)

print('Agent listo.')

## 7. Loop de entrenamiento

In [ ]:
from collections import deque
from pathlib import Path
import time

def train(cfg, total_steps=300_000, log_every=2_000, checkpoint_every=50_000,
          save_dir='saves', target_solved=200.0):
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    env = gym.make(cfg.env_id)
    agent = DQNAgent(cfg)

    episode_returns = []
    log_steps, log_running, log_eps, log_loss = [], [], [], []
    running = deque(maxlen=100)
    recent_losses = deque(maxlen=1000)

    obs, _ = env.reset(seed=cfg.seed)
    ep_return = 0.0; ep_length = 0
    t0 = time.time()
    solved_at = None

    for step in range(1, total_steps + 1):
        a = agent.select_action(obs)
        next_obs, r, term, trunc, _ = env.step(a)
        agent.buffer.push(obs, a, float(r), next_obs, term)
        obs = next_obs
        ep_return += float(r); ep_length += 1
        agent.step_count = step

        if step % cfg.learn_every == 0:
            loss = agent.learn()
            if loss is not None: recent_losses.append(loss)

        if step % cfg.target_update_every == 0:
            agent.sync_target()

        if term or trunc:
            episode_returns.append(ep_return); running.append(ep_return)
            ep_return = 0.0; ep_length = 0
            obs, _ = env.reset()

        if step % log_every == 0:
            eps = agent.epsilon()
            rmean = float(np.mean(running)) if running else float('nan')
            avg_loss = float(np.mean(recent_losses)) if recent_losses else float('nan')
            log_steps.append(step); log_running.append(rmean)
            log_eps.append(eps); log_loss.append(avg_loss)
            elapsed = time.time() - t0
            fps = step / max(elapsed, 1e-9)
            marker = ''
            if rmean >= target_solved and solved_at is None:
                solved_at = step
                marker = ' ★ SOLVED (mean100 ≥ 200)'
            print(f'step {step:>7} | ep {len(episode_returns):>4} | '
                  f'return(mean100) {rmean:>+7.2f} | eps {eps:.3f} | '
                  f'loss {avg_loss:.4f} | buffer {len(agent.buffer):>6} | '
                  f'{fps:.0f} step/s | {elapsed/60:.1f} min{marker}', flush=True)

        if step % checkpoint_every == 0:
            agent.save(f'{save_dir}/lunarlander_dqn_{step}.pt')
            np.savez(f'{save_dir}/history.npz',
                     episode_returns=np.array(episode_returns, dtype=np.float32),
                     log_steps=np.array(log_steps, dtype=np.int64),
                     log_running_mean=np.array(log_running, dtype=np.float32),
                     log_epsilon=np.array(log_eps, dtype=np.float32),
                     log_loss=np.array(log_loss, dtype=np.float32),
                     solved_at=solved_at if solved_at else -1)

    agent.save(f'{save_dir}/lunarlander_dqn_final.pt')
    np.savez(f'{save_dir}/history.npz',
             episode_returns=np.array(episode_returns, dtype=np.float32),
             log_steps=np.array(log_steps, dtype=np.int64),
             log_running_mean=np.array(log_running, dtype=np.float32),
             log_epsilon=np.array(log_eps, dtype=np.float32),
             log_loss=np.array(log_loss, dtype=np.float32),
             solved_at=solved_at if solved_at else -1)
    env.close()
    return agent, solved_at

## 8. Entrenar 🚀

En T4 el ritmo esperado es ~800–1500 step/s. 300 000 pasos ≈ **~5–10 min**.

El entrenamiento se detiene automáticamente cuando alcanza el criterio de "solved" (retorno medio ≥ 200 en 100 eps) — el resto del tiempo es para consolidar.

In [ ]:
cfg = DQNConfig()
agent, solved_at = train(cfg, total_steps=300_000, log_every=2_000,
                          checkpoint_every=50_000, save_dir=DRIVE_DIR)
print(f'\nEntrenamiento completo. Solved en step: {solved_at}')
print(f'Artefactos en: {DRIVE_DIR}')

## 9. Curva de aprendizaje

In [ ]:
import matplotlib.pyplot as plt

h = np.load(f'{DRIVE_DIR}/history.npz')
returns = h['episode_returns']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

ax1.plot(returns, color='steelblue', alpha=0.25, linewidth=0.6, label='Retorno por episodio')
win = 20
if len(returns) >= win:
    smooth = np.convolve(returns, np.ones(win)/win, mode='valid')
    ax1.plot(np.arange(win-1, len(returns)), smooth, color='firebrick', linewidth=2,
             label=f'Media móvil ({win} eps)')
ax1.axhline(200, color='green', linestyle='--', alpha=0.7, label='Umbral solved (+200)')
ax1.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Episodio')
ax1.set_ylabel('Retorno acumulado')
ax1.set_title('LunarLander-v3 · DQN — Curva de aprendizaje')
ax1.legend(loc='lower right', framealpha=0.9, fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(h['log_steps'], h['log_running_mean'], color='steelblue', linewidth=2,
         label='Retorno medio (100 eps)')
ax2.axhline(200, color='green', linestyle='--', alpha=0.6, label='Umbral solved')
ax2.set_xlabel('Pasos del agente')
ax2.set_ylabel('Retorno medio (100 eps)', color='steelblue')
ax2.grid(True, alpha=0.3)
ax2b = ax2.twinx()
ax2b.plot(h['log_steps'], h['log_epsilon'], color='orange', linewidth=1.5, alpha=0.75, label='ε')
ax2b.set_ylabel('ε', color='orange')
ax2.set_title('Retorno vs. ε en el tiempo')

plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/lunarlander_learning_curve.png', dpi=140, bbox_inches='tight')
plt.show()

print(f'Total episodios: {len(returns)}')
if len(returns) >= 100:
    print(f'Retorno medio (últimos 100 eps): {returns[-100:].mean():+.2f}')
print(f'Retorno máximo alcanzado: {returns.max():+.2f}')
print(f'Solved en step: {int(h["solved_at"]) if h["solved_at"] > 0 else "NO"}')

## 10. Evaluación con política voraz

In [ ]:
def evaluate(agent, n_episodes=20, seed_offset=10_000):
    env = gym.make('LunarLander-v3')
    returns = []
    landings = 0
    for i in range(n_episodes):
        obs, _ = env.reset(seed=seed_offset + i)
        R, done = 0.0, False
        while not done:
            a = agent.select_action(obs, deterministic=True)
            obs, r, term, trunc, _ = env.step(a)
            R += float(r); done = term or trunc
        returns.append(R)
        if R >= 200: landings += 1
    env.close()
    return np.array(returns), landings

eval_returns, landings = evaluate(agent, n_episodes=20)
print(f'=== Evaluación (20 eps, greedy) ===')
print(f'  media: {eval_returns.mean():+.2f}')
print(f'  std:   {eval_returns.std():.2f}')
print(f'  min:   {eval_returns.min():+.2f}')
print(f'  max:   {eval_returns.max():+.2f}')
print(f'  aterrizajes exitosos (retorno ≥ 200): {landings}/{len(eval_returns)}')
print(f'\nDetalle: {[float(f"{r:+.1f}") for r in eval_returns]}')

np.savez(f'{DRIVE_DIR}/eval_results.npz', returns=eval_returns, landings=landings)

## 11. Video del agente aterrizando

In [ ]:
import imageio

def record_gameplay(agent, out_path, n_episodes=3):
    env = gym.make('LunarLander-v3', render_mode='rgb_array')
    all_frames = []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=12_345 + i)
        done = False
        while not done:
            all_frames.append(env.render())
            a = agent.select_action(obs, deterministic=True)
            obs, r, term, trunc, _ = env.step(a)
            done = term or trunc
    env.close()
    imageio.mimsave(out_path, all_frames, fps=30)
    print(f'Video guardado: {out_path} ({len(all_frames)} frames)')

record_gameplay(agent, f'{DRIVE_DIR}/lunarlander_gameplay.mp4', n_episodes=3)

## 12. Descargar artefactos

In [ ]:
from google.colab import files

for f in ['lunarlander_dqn_final.pt', 'history.npz', 'eval_results.npz',
          'lunarlander_learning_curve.png', 'lunarlander_gameplay.mp4']:
    path = f'{DRIVE_DIR}/{f}'
    if os.path.exists(path):
        files.download(path)
        print(f'Descargando {f}')